# QuickCart: Data Understanding

Initial inspection of the raw quick-commerce orders dataset for the QuickCart Decision Intelligence System.

This notebook validates dataset structure, data quality, key business fields, and early analysis opportunities before cleaning, SQL analysis, Power BI dashboarding, and decision intelligence development.


## Setup

In [41]:
import pandas as pd
import numpy as np

## Load & Preview Data

In [42]:
df = pd.read_csv("quickcart_raw_orders.csv")

In [43]:
df.head()

,Order ID,Customer ID,Platform,Order Date & Time,Delivery Time (Minutes),Product Category,Order Value (INR),Customer Feedback,Service Rating,Delivery Delay,Refund Requested
0,ORD000001,CUST2824,JioMart,19:29.5,30,Fruits & Vegetables,382,"Fast delivery, great service!",5,No,No
1,ORD000002,CUST1409,Blinkit,54:29.5,16,Dairy,279,Quick and reliable!,5,No,No
2,ORD000003,CUST5506,JioMart,21:29.5,25,Beverages,599,Items missing from order.,2,No,Yes
3,ORD000004,CUST5012,JioMart,19:29.5,42,Beverages,946,Items missing from order.,2,Yes,Yes
4,ORD000005,CUST4657,Blinkit,49:29.5,30,Beverages,334,"Fast delivery, great service!",5,No,No


**Observation:** Each row represents an individual quick-commerce order with fields related to platform, customer, delivery time, product category, order value, customer feedback, service rating, delay status, and refund request.

## Dataset Overview

In [44]:
df.shape

(100000, 11)

**Observation:** The dataset contains 100,000 rows and 11 columns, making it large enough for meaningful operational, customer experience, and platform-level analysis.

In [45]:
list(df.columns)

['Order ID',
 'Customer ID',
 'Platform',
 'Order Date & Time',
 'Delivery Time (Minutes)',
 'Product Category',
 'Order Value (INR)',
 'Customer Feedback',
 'Service Rating',
 'Delivery Delay',
 'Refund Requested']

**Observation:** The dataset includes order identifiers, customer identifiers, platform, delivery metrics, product category, order value, customer feedback, rating, delay status, and refund status.

In [46]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 11 columns):
 #   Column                   Non-Null Count   Dtype 
---  ------                   --------------   ----- 
 0   Order ID                 100000 non-null  object
 1   Customer ID              100000 non-null  object
 2   Platform                 100000 non-null  object
 3   Order Date & Time        100000 non-null  object
 4   Delivery Time (Minutes)  100000 non-null  int64 
 5   Product Category         100000 non-null  object
 6   Order Value (INR)        100000 non-null  int64 
 7   Customer Feedback        100000 non-null  object
 8   Service Rating           100000 non-null  int64 
 9   Delivery Delay           100000 non-null  object
 10  Refund Requested         100000 non-null  object
dtypes: int64(3), object(8)
memory usage: 8.4+ MB


**Observation:** The dataset contains 3 numeric columns and 8 object-type columns. Delivery time, order value, and service rating are already numeric. The `Order Date & Time` column is stored as text and will require further review before time-based analysis.

## Data Quality Checks

In [47]:
missing_summary = pd.DataFrame({
    "missing_values": df.isnull().sum(),
    "missing_percentage": (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values(by="missing_values", ascending=False)

missing_summary

,missing_values,missing_percentage
Order ID,0,0.0
Customer ID,0,0.0
Platform,0,0.0
Order Date & Time,0,0.0
Delivery Time (Minutes),0,0.0
Product Category,0,0.0
Order Value (INR),0,0.0
Customer Feedback,0,0.0
Service Rating,0,0.0
Delivery Delay,0,0.0


In [48]:
duplicate_count = df.duplicated().sum()
duplicate_percentage = round((duplicate_count / len(df)) * 100, 2)

duplicate_count, duplicate_percentage

(np.int64(0), np.float64(0.0))

**Observation:** The dataset has no missing values and no duplicate rows. This confirms that the raw dataset is structurally clean at the basic quality-check level.

## Descriptive Summary

In [49]:
df.describe()

,Delivery Time (Minutes),Order Value (INR),Service Rating
count,100000.000000,100000.000000,100000.000000
mean,29.536140,590.994400,3.240790
std,9.958933,417.409058,1.575962
min,5.000000,50.000000,1.000000
25%,23.000000,283.000000,2.000000
50%,30.000000,481.000000,3.000000
75%,36.000000,770.000000,5.000000
max,76.000000,2000.000000,5.000000


In [50]:
df.describe(include="object")

,Order ID,Customer ID,Platform,Order Date & Time,Product Category,Customer Feedback,Delivery Delay,Refund Requested
count,100000,100000,100000,100000,100000,100000,100000,100000
unique,100000,9000,3,60,6,13,2,2
top,ORD099984,CUST8779,Swiggy Instamart,50:29.5,Dairy,"Easy to order, loved it!",No,No
freq,1,26,33449,1755,16857,7791,86328,54181


**Observation:** The dataset contains three key numeric fields: delivery time, order value, and service rating.

Average delivery time is approximately 29.54 minutes, average order value is around ₹591, and average service rating is 3.24.

The categorical fields show 100,000 unique order IDs, 9,000 unique customers, 3 platforms, 6 product categories, 13 customer feedback types, and binary delay/refund indicators.

## Column Summary

In [51]:
column_summary = pd.DataFrame({
    "column_name": df.columns,
    "data_type": df.dtypes.values,
    "missing_values": df.isnull().sum().values,
    "missing_percentage": (df.isnull().sum().values / len(df) * 100).round(2),
    "unique_values": df.nunique().values
})

column_summary

,column_name,data_type,missing_values,missing_percentage,unique_values
0,Order ID,object,0,0.0,100000
1,Customer ID,object,0,0.0,9000
2,Platform,object,0,0.0,3
3,Order Date & Time,object,0,0.0,60
4,Delivery Time (Minutes),int64,0,0.0,69
5,Product Category,object,0,0.0,6
6,Order Value (INR),int64,0,0.0,1951
7,Customer Feedback,object,0,0.0,13
8,Service Rating,int64,0,0.0,5
9,Delivery Delay,object,0,0.0,2


**Observation:** The dataset has no missing values across all columns. `Order ID` is unique for every record, while `Customer ID` appears multiple times, supporting repeat-customer analysis.

The available fields are sufficient for platform comparison, delivery performance analysis, customer experience analysis, refund behavior analysis, KPI monitoring, and decision intelligence development.

## Business Field Distribution

In [52]:
important_categorical_cols = [
    "Platform",
    "Product Category",
    "Customer Feedback",
    "Service Rating",
    "Delivery Delay",
    "Refund Requested"
]

distribution_rows = []

for col in important_categorical_cols:
    counts = df[col].value_counts()
    percentages = (df[col].value_counts(normalize=True) * 100).round(2)

    for value in counts.index:
        distribution_rows.append({
            "column_name": col,
            "category": value,
            "count": counts[value],
            "percentage": percentages[value]
        })

business_distribution = pd.DataFrame(distribution_rows)

business_distribution

,column_name,category,count,percentage
0,Platform,Swiggy Instamart,33449,33.45
1,Platform,Blinkit,33424,33.42
2,Platform,JioMart,33127,33.13
3,Product Category,Dairy,16857,16.86
4,Product Category,Grocery,16737,16.74
5,Product Category,Snacks,16705,16.71
6,Product Category,Fruits & Vegetables,16632,16.63
7,Product Category,Beverages,16536,16.54
8,Product Category,Personal Care,16533,16.53
9,Customer Feedback,"Easy to order, loved it!",7791,7.79


**Observation:** Platform and product category distributions are almost balanced, which supports fair comparison across platforms and categories.

The delivery delay rate is 13.67%, while the refund request rate is 45.82%. Refund requests are much higher than delivery delays, suggesting that refunds may also be linked to missing items, wrong items, product quality issues, low ratings, or negative feedback.

Service ratings appear polarized, with 5-star ratings being the most common, followed by 2-star and 1-star ratings.

## Identifier Validation

In [53]:
identifier_summary = pd.DataFrame({
    "column_name": ["Order ID", "Customer ID"],
    "is_unique": [df["Order ID"].is_unique, df["Customer ID"].is_unique],
    "unique_values": [df["Order ID"].nunique(), df["Customer ID"].nunique()]
})

identifier_summary

,column_name,is_unique,unique_values
0,Order ID,True,100000
1,Customer ID,False,9000


## Customer Order Frequency

In [54]:
customer_order_frequency = df["Customer ID"].value_counts()

customer_frequency_metrics = pd.DataFrame({
    "metric": [
        "Unique Customers",
        "Average Orders per Customer",
        "Median Orders per Customer",
        "Minimum Orders by a Customer",
        "Maximum Orders by a Customer"
    ],
    "value": [
        customer_order_frequency.count(),
        round(customer_order_frequency.mean(), 2),
        customer_order_frequency.median(),
        customer_order_frequency.min(),
        customer_order_frequency.max()
    ]
})

customer_frequency_metrics

,metric,value
0,Unique Customers,9000.00
1,Average Orders per Customer,11.11
2,Median Orders per Customer,11.00
3,Minimum Orders by a Customer,2.00
4,Maximum Orders by a Customer,26.00


In [55]:
top_repeat_customers = customer_order_frequency.head(10).reset_index()
top_repeat_customers.columns = ["customer_id", "order_count"]

top_repeat_customers

,customer_id,order_count
0,CUST8779,26
1,CUST1848,25
2,CUST8289,24
3,CUST3532,23
4,CUST3001,23
5,CUST2633,23
6,CUST5902,23
7,CUST7557,22
8,CUST1897,22
9,CUST5476,22


In [56]:
def customer_frequency_segment(order_count):
    if order_count <= 5:
        return "Low Frequency"
    elif order_count <= 15:
        return "Medium Frequency"
    else:
        return "High Frequency"

customer_frequency_segments = customer_order_frequency.apply(customer_frequency_segment)
customer_frequency_counts = customer_frequency_segments.value_counts()

customer_frequency_summary = pd.DataFrame({
    "frequency_segment": customer_frequency_counts.index,
    "customer_count": customer_frequency_counts.values
})

customer_frequency_summary["customer_percentage"] = (
    customer_frequency_summary["customer_count"] / customer_frequency_summary["customer_count"].sum() * 100
).round(2)

customer_frequency_summary

,frequency_segment,customer_count,customer_percentage
0,Medium Frequency,7777,86.41
1,High Frequency,910,10.11
2,Low Frequency,313,3.48


**Observation:** `Order ID` is unique, confirming that each row represents one order. `Customer ID` is not unique, indicating repeat-order behavior.

The dataset contains 9,000 unique customers, with an average of 11.11 orders per customer. The highest-frequency customer placed 26 orders.

Most customers fall into the Medium Frequency segment, representing 86.41% of the customer base. High Frequency customers represent 10.11%, making them useful for customer engagement and retention-related analysis.

## Numeric Value Validation

In [57]:
invalid_numeric_summary = pd.DataFrame({
    "validation_check": [
        "Delivery time <= 0",
        "Order value <= 0",
        "Service rating outside 1-5"
    ],
    "invalid_records": [
        invalid_delivery_time,
        invalid_order_value,
        invalid_service_rating
    ]
})

invalid_numeric_summary

,validation_check,invalid_records
0,Delivery time <= 0,0
1,Order value <= 0,0
2,Service rating outside 1-5,0


**Observation:** Delivery time ranges from 5 to 76 minutes, order value ranges from ₹50 to ₹2000, and service rating ranges from 1 to 5.

No invalid values were found in the key numeric fields, confirming that these columns are suitable for KPI calculation and decision intelligence analysis.

In [58]:
binary_validation = pd.DataFrame({
    "field": ["Delivery Delay", "Refund Requested"],
    "unique_values": [
        list(df["Delivery Delay"].unique()),
        list(df["Refund Requested"].unique())
    ],
    "yes_count": [
        (df["Delivery Delay"] == "Yes").sum(),
        (df["Refund Requested"] == "Yes").sum()
    ],
    "yes_percentage": [
        round((df["Delivery Delay"] == "Yes").mean() * 100, 2),
        round((df["Refund Requested"] == "Yes").mean() * 100, 2)
    ]
})

binary_validation

,field,unique_values,yes_count,yes_percentage
0,Delivery Delay,"[No, Yes]",13672,13.67
1,Refund Requested,"[No, Yes]",45819,45.82


**Observation:** Both binary fields contain only `Yes` and `No` values.

The delivery delay rate is 13.67%, while the refund request rate is 45.82%. The higher refund rate suggests that refund behavior should be investigated beyond delivery delays.

## Time Column Review

In [59]:
df["Order Date & Time"].head(10)

,Order Date & Time
0,19:29.5
1,54:29.5
2,21:29.5
3,19:29.5
4,49:29.5
5,36:29.5
6,22:29.5
7,50:29.5
8,51:29.5
9,08:29.5


In [60]:
df["Order Date & Time"].nunique()

60

In [61]:
time_format_check = df["Order Date & Time"].astype(str).str.match(r"^\d{2}:\d{2}\.\d$")

time_format_check.value_counts()

,count
Order Date & Time,
True,100000


**Observation:** The `Order Date & Time` field has 60 unique values and follows a consistent format across all records.

However, values such as `54:29.5` and `58:29.5` do not represent standard clock time. This field should be reviewed during data cleaning before using it for time-based analysis.

## Platform Performance Overview

In [62]:
platform_snapshot = df.groupby("Platform").agg(
    total_orders=("Order ID", "count"),
    avg_delivery_time=("Delivery Time (Minutes)", "mean"),
    avg_order_value=("Order Value (INR)", "mean"),
    avg_service_rating=("Service Rating", "mean")
).round(2)

platform_snapshot

,total_orders,avg_delivery_time,avg_order_value,avg_service_rating
Platform,,,,
Blinkit,33424,29.47,589.55,3.23
JioMart,33127,29.63,590.53,3.25
Swiggy Instamart,33449,29.50,592.90,3.24


In [63]:
platform_risk_snapshot = df.groupby("Platform").agg(
    delay_rate=("Delivery Delay", lambda x: (x == "Yes").mean() * 100),
    refund_rate=("Refund Requested", lambda x: (x == "Yes").mean() * 100)
).round(2)

platform_risk_snapshot

,delay_rate,refund_rate
Platform,,
Blinkit,13.38,45.93
JioMart,13.83,45.82
Swiggy Instamart,13.81,45.71


**Observation:** All three platforms have similar order volumes, average delivery times, average order values, and service ratings.

Blinkit has the lowest delay rate at 13.38%, while JioMart has the highest at 13.83%. Refund request rates are high across all platforms, with each platform showing a refund rate of around 46%.

At this stage, platform performance appears broadly similar, but refund behavior and customer experience require deeper analysis in the next phases.


## Initial Data Understanding Scorecard

In [64]:
initial_scorecard = pd.DataFrame({
    "metric": [
        "Total Orders",
        "Total Columns",
        "Missing Values",
        "Duplicate Rows",
        "Unique Customers",
        "Average Delivery Time",
        "Average Order Value",
        "Average Service Rating",
        "Delivery Delay Rate",
        "Refund Request Rate",
        "Top Platform by Orders",
        "Top Product Category",
        "Time Field Status"
    ],
    "value": [
        df.shape[0],
        df.shape[1],
        df.isnull().sum().sum(),
        df.duplicated().sum(),
        df["Customer ID"].nunique(),
        round(df["Delivery Time (Minutes)"].mean(), 2),
        round(df["Order Value (INR)"].mean(), 2),
        round(df["Service Rating"].mean(), 2),
        str(round((df["Delivery Delay"] == "Yes").mean() * 100, 2)) + "%",
        str(round((df["Refund Requested"] == "Yes").mean() * 100, 2)) + "%",
        df["Platform"].value_counts().idxmax(),
        df["Product Category"].value_counts().idxmax(),
        "Consistent format, non-standard time value"
    ]
})

initial_scorecard

,metric,value
0,Total Orders,100000
1,Total Columns,11
2,Missing Values,0
3,Duplicate Rows,0
4,Unique Customers,9000
5,Average Delivery Time,29.54
6,Average Order Value,590.99
7,Average Service Rating,3.24
8,Delivery Delay Rate,13.67%
9,Refund Request Rate,45.82%


**Observation:** The dataset is structurally clean and ready for the next phase. The main issues identified are the high refund request rate, the non-standard `Order Date & Time` format, and the polarized service rating distribution.
